In [0]:
%pip install openai unidecode gspread==5.12.4

In [0]:
%restart_python
%load_ext autoreload
%autoreload 2 

In [0]:
import pandas as pd
import json
import random
import openai
from openai import OpenAI
from pathlib import Path
from unidecode import unidecode
import gspread

from pyspark.sql.functions import *
import pyspark
from functools import reduce

import time

In [0]:
from config import *

In [0]:
%run "./authentication script"

In [0]:
ASO_RESPONSES_ID = "1gDZMrFiPVvOosjrA0wJsYzMOj1HBBGOcpwujS3NuD3c"
# RESPONSES_TAB = "Form Responses 1"
# RESPONSES_HEADERS = "A1:G1"
# RESPONSES_DATA_FORMAT = "A2:G"

In [0]:
ASO_PROCESSING_WORKBOOK_ID = "1WnCBGk1V1nEUeU7aPWBqrQXczmaMop6AbGmkDKkAQcE"
#TAB = "Submissions"

#SLUG_COLS = ["A","B","C","D","E","F","G"]
#INPUTS_WORKBOOK_ID_COL = "T"

In [0]:
gc=gspread.service_account_from_dict(json.loads(crafty_json))

In [0]:
submissions_sh = gc.open_by_key(ASO_PROCESSING_WORKBOOK_ID) # gspread worksheet object
submissions = submissions_sh.worksheet("Submissions")

In [0]:
# Get entries data 
entries = submissions.get("A2:T")
submission_header_cols = submissions.get("A1:T1")[0]


# As of 5/28/25:
# submission_header_cols = ['Timestamp', 'Submitter Email', 'Due Date', 'Game', 'Target Languages', 'URL', 'QA Flag', 'Permission Status', 'Share Status', 'Data Valid', 'Internal Status', 'Ready For Job', 'Job ID', 'Result Sheet URL', 'Synced At', 'Job Launched At', 'Job Completed At', 'Last Updated At', 'Row Fingerprint', 'Workbook ID']


In [0]:
pd_entries = pd.DataFrame(entries, columns=submission_header_cols)

In [0]:
#def create_results_sheet(gc):
#   title = "" 
#   sh = gc.create(title)
#   sh.share(share_email)
#   sh.share(submitter_email)
#   return sheet.url # or workbook_id

#def create_proofreading_sheet(gc, target_languages):
#   title = "" 
#   sh = gc.create(title)
#   for language in target_languages:
#       worksheets = sh.add_worksheet(title=language)
#   sh.share(share_email)
#   sh.share(submitter_email)
#   sh.share("leila.bradaran@jamcity.com") # For proofreading, for now; 
#   return sheet.url # or workbook_id

#TODO:
- check for new updates to table and write to table
- only process new rows 
- GPT API transition (from batch to chat completions.) (formatting first, then kicking the job off)
- Passing results back to a sheet and sending an email/slack notification
- Formatting for Proofreading (if QA_Flag = True);
- Formatting for final copy 

In [0]:
# DATABRICKS ASO TABLE 
#entries_df = spark.createDataFrame(entries, submission_header_cols)
# TODO: Want to update this as a delta table? or just insert and append
aso_monthly_submissions_table = 'ds.aso_monthly_submissions'
#entries_df.write....

In [0]:
def get_data_from_input(spreadsheet, platform, game):
    worksheet = spreadsheet.worksheet(platform)
    if platform == 'android':
        data = worksheet.get("A4:B")
        df = pd.DataFrame(data,columns = ['en_US_80','en_US_500'])
        df['game'] = game
        df['platform'] = platform
        df['row_id'] = df.index
    if platform == 'ios':
        data = worksheet.get("A4:C")
        df = pd.DataFrame(data, columns = ['en_US_30','en_US_50','en_US_120'])
        df['game'] = game
        df['platform'] = platform
        df['row_id'] = df.index
    
    if df.empty: 
        return None
    return df

In [0]:
#['Timestamp', 'Submitter Email', 'Due Date', 'Game', 'Target Languages', 'URL', 'QA Flag', 'Permission Status', 'Share Status', 'Data Valid', 'Internal Status', 'Ready For Job', 'Job ID', 'Result Sheet URL', 'Synced At', 'Job Launched At', 'Job Completed At', 'Last Updated At', 'Row Fingerprint', 'Workbook ID']


In [0]:
def process_row_for_job_submission(row):
    url = row['URL']
    game = row['Game']

    #TODO: Put this in a try except block; if it fails we need to do something
    try:
        sh = gc.open_by_url(url)
        permission_flag = "Success!"
        inputs_holder = {
            'context':
                {
                    'game': game,
                    'url':url,
                    'qa_flag':row['QA Flag'],
                    'target_languages': row['Target Languages'].split(','),
                    'submitter_email': row['Submitter Email'],
                    'submission_timestamp':row['Timestamp'],
                    'due_date':row['Due Date'],
                    'permission_flag':permission_flag,
                }
        }
    except:
        print(f"Couldn't open {url}")
        permission_flag = 'Denied!'
        inputs_holder = {
        'context':
            {
                'game': game,
                'url':url,
                'qa_flag':row['QA Flag'],
                'target_languages': row['Target Languages'].split(','),
                'submitter_email': row['Submitter Email'],
                'submission_timestamp':row['Timestamp'],
                'due_date':row['Due Date'],
                'permission_flag':permission_flag,
            }
        }
        return inputs_holder

    # Process the data from the sheet
    ios_inputs_df = get_data_from_input(sh, 'ios', game)
    android_inputs_df = get_data_from_input(sh, 'android', game)
    ios_long_inputs = convert_inputs_to_long(ios_inputs_df,'ios')
    android_long_inputs = convert_inputs_to_long(android_inputs_df,'android')
    inputs_holder['ios'] = {
                'inputs':ios_inputs_df, 
                'long_inputs':ios_long_inputs
            }
    inputs_holder['android'] = {
                'inputs': android_inputs_df,
                'long_inputs':android_long_inputs
            }

    return inputs_holder

In [0]:
q_ios = f"""
        select 
            game,
            platform,
            row_id,
            'title' as type_desc,
            en_US_30 as en_US,
            30 as en_char_limit
        from df
        union all
        select 
            game,
            platform,
            row_id,
            'short_description' as type_desc,
            en_US_50 as en_US,
            50 as en_char_limit
        from df
        union all
        select 
            game,
            platform,
            row_id,
            'long_description' as type_desc,
            en_US_120 as en_US,
            120 as en_char_limit
        from df
"""


#TODO: TEST THIS
q_android= f""" select 
            game,
            year_month,
            platform,
            row_id,
            'short_description' as type_desc,
            en_US_80 as en_US,
            80 as en_char_limit
        from df
        union all
        select 
            game,
            year_month,
            platform,
            row_id,
            'long_description' as type_desc,
            en_US_500 as en_US,
            500 as en_char_limit
        from df
    """



def convert_inputs_to_long(df: pd.DataFrame, 
                           platform:str)->pd.DataFrame:
    # TODO: first test to see if empty, 
    if df is None:
        return None
    spark.createDataFrame(df).createOrReplaceTempView('df')
    if platform == 'ios':
        long_df = spark.sql(q_ios).toPandas()
    if platform == 'android':
        long_df = spark.sql(q_android).toPandas()

    return long_df

In [0]:
#pd_entries[0:1]

In [0]:
# Ok, process the entries
## TODO: Make sure the pd_entries are only new ones...
submissions_holder = []
for idx, row in pd_entries.iterrows():
    submissions_holder.append(process_row_for_job_submission(row))


In [0]:
TARGET_LANGUAGE_MAPPING = {
    'Spanish (Latin America)':'es_LA',
    ' French (France)':'fr_FR',
    ' German':'de_DE',
    ' Russian':'ru_RU',
    ' Korean':'ko_KR',
    ' Italian':'it_IT',
    ' Japanese': 'ja_JP',
    ' Simplified Chinese': 'zh_CN',
    ' Traditional Chinese (Taiwan)':'zh_TW',
    ' Portuguese (Brazil)':'pt_BR',
}

In [0]:
def add_lang_and_char_limits(input_df:pd.DataFrame,
                             language:str,
                             platform:str)->pd.DataFrame:
    language_cd  = TARGET_LANGUAGE_MAPPING[language]
    print(f'received language..{language} with {language_cd}')
    df = input_df.copy()
    if language_cd in ['ja_JP','ko_KR','zh_CN','zh_TW'] and platform == 'android':
         df['target_char_limit'] = int(df['en_char_limit']/2)
    else:
        df['target_char_limit'] = df['en_char_limit']
    df['language'] = language
    df['language_cd'] = language_cd

    #TODO: Add uniqiue identifier here too
    #df['row_unique_idx'] = f"row_{df['game']}_{df['platform']}_{df['language_cd']}_{df['row_id']}"
    df["row_idx"] = df.apply(
        lambda row: f"row_{row['row_id']}::{row['game'].replace(' ', '')}::{row['platform']}::{row['language_cd']}",
        axis=1
    )
   
    return df

        

In [0]:
#def format_prompt(df,language,platform):


In [0]:
def format_all_languages(submission):
    all_languages_holder = []
    if submission['context']['permission_flag']=='Denied!':
        return all_languages_holder
    target_languages = submission['context']['target_languages']
    
    for platform in ['ios','android']:
        if submission[platform]['inputs']is None:
            print('not processing this data...')
        else:
            for language in target_languages:
                print(f'format all languages by language for {language} and {platform}')
                data = submission[platform]['long_inputs']
                by_language = add_lang_and_char_limits(data,language, platform)

                #TODO: Actually, add an identifier to the by_language df
                all_languages_holder.append(by_language)

                #TODO: call the GPT function here with (by_language,language,platform,game)

    return all_languages_holder 

In [0]:
#formatted_inputs 

formatted_inputs = format_all_languages(submissions_holder[0])

In [0]:
ALL_LANGS = ['Spanish (Latin America)',
 ' French (France)',
 ' German',
 ' Russian',
 ' Korean',
 ' Italian',
 ' Japanese',
 ' Simplified Chinese',
 ' Traditional Chinese (Taiwan)',
 ' Portuguese (Brazil)']



In [0]:


LANG_SPECIFIC_GUIDELINES = {
    'Spanish (Latin America)':""" - Use informal tú-form - Prioritize friendly, casual verbs like juega, descubre, gana""",
    ' Portuguese (Brazil)': """ - Use informal você-form - Make copy energetic and emotionally expressive — Divirta-se!""",
    ' Italian': """ -Use informal tone with playful verbs like Gioca, Scopri, Divertiti """,
    ' Japanese': """ - Use casual-polite forms (e.g., ～しよう, ～が登場), - Match the upbeat, punchy tone of puzzle and gacha games """,
    ' French (France)':""" -Use informal tu-form -Make phrasing smooth, vivid, and naturally expressive """,
    ' German': """ """,
    ' Simplified Chinese': """ - Keep it brief, casual, and direct - Highlight excitement and rewards with punchy terms like 限时, 赢奖励 """, 
    ' Traditional Chinese (Taiwan)': """ - Use casual and lively language that fits mobile game audiences in Taiwan and Hong Kong
    -Favor clear, short sentences with a playful or promotional tone
    -Prioritize fluency and cultural appropriateness over literal phrasing
    -Terms like 表情符號 (emoji), 獎勵 (rewards), and 限時 (limited-time) are common in game copy
    -Avoid overly technical or overly simplified language — it should feel local and fun
    """, 
    ' Korean':""" - Favor casual or semi-formal style depending on context - Keep copy concise, lively, and visually engaging """,
    ' Russian': """ -Always use the formal second-person plural (вы) 
    -Do not capitalize вы — this is a neutral formal register, not overly honorific
    -All verbs and adjectives must match this formal second-person form
    -The tone should be polite but not stiff or bureaucratic
    -Make phrasing natural and suitable for a general gaming audience"""
}


In [0]:
GENERAL_GUIDELINES = f""" General Translation Guidelines:


The tone should be:

-Fun, playful, and energetic

-Casual and approachable

-Clear, concise, and engaging for players

You must:

- Use natural, idiomatic language for the target audience

- Prioritize clarity and emotional appeal over literal translation

- Maintain consistent tone and phrasing across all content

- Stay within any provided character limits

If the English text includes puns, idioms, or culturally specific phrases:

- Adapt them to something that feels native and engaging in the target language

- It's okay to rephrase for clarity or punch — fun and fluid is better than literal

Avoid:

- Overly formal or technical phrasing

- Translating idioms or jokes literally if they don’t work in the target language
"""




In [0]:
GAME_SPECIFIC_GUIDELINES = {
    'Cookie Jam':'A match-3 game that features themes of baking, and a Chef Panda as the main character.',
    'Disney Emoji Blitz':'A game that takes place in the Disney universe - so you may see some references to characters in the now scope of the Disney franchise. Please maintain consistency to the already established translations of Disney related content. ',
    'Panda Pop':'A bubble shooter game that involves a Mama Panda and her numerous baby pandas.',
    'Harry Potter: A Hogwarts Mystery':""" Harry Potter: A Hogwarts Mystery\n Game premise: A mobile narrative RPG set at Hogwarts in the 1980s. Players create their own witch or wizard, attend classes, build friendships, and take part in limited-time quests inspired by seasons, holidays, and Wizarding World lore.

    Tone: Keep copy short, hype-driven, and “magical” (e.g., celebrate, unlock, nurture, discover) while sounding warm and inclusive.

    Proper nouns & spells: Preserve every franchise term exactly as it appears in the official localized Wizarding World canon for your language. Examples: Hufflepuff → Poufsouffle (FR), Diagon Alley → Chemin de Traverse (FR). Do not alter house names, character names, spells, or location names.

    What can change: Ordinary adjectives, verbs, and connective text may be freely adapted to fit local style and store character limits.""",# TODO: ADD HPHM GAME
    } 

In [0]:
GENERAL_GUIDELINES = f""" General Translation Guidelines:

The tone should be:
- Fun, playful, and energetic
- Casual and approachable
- Clear, concise, and engaging for players

You must:
- Use natural, idiomatic language for the target audience
- Prioritize clarity and emotional appeal over literal translation
- Maintain consistent tone and phrasing across all content
- Stay within any provided character limits

If the English text includes puns, idioms, or culturally specific phrases:
- Adapt them to something that feels native and engaging in the target language
- It's okay to rephrase for clarity or punch — fun and fluid is better than literal

Avoid:
- Overly formal or technical phrasing
- Translating idioms or jokes literally if they don’t work in the target language
"""

In [0]:
def build_translation_prompt(game:str, 
                             language:str)->str:
    
    game_specific = GAME_SPECIFIC_GUIDELINES[game]
    language_specific = LANG_SPECIFIC_GUIDELINES[language]
    return f""""

    You are a professional game copy localizer.

    Translate the following English phrases into Spanish (Latin America) for a mobile game. Use the context and tone described below. Ensure each translation is idiomatic, natural, and within the specified character limit.

    --- Guidelines ---
    {GENERAL_GUIDELINES}

    --- Context ---
    Game: {game}
    Game Description: {game_specific}

    
    --- Language Notes ---
    {language_specific}

    --- Instructions ---
    - Use the game context and tone described above.  
    - Each phrase must be translated idiomatically and playfully.  
    - Do **NOT** exceed the provided character limit.  
    - Return results in JSON format.

    --- Output Format ---
    Return **only** a valid JSON object with the following keys and values:
    - row_idx --> row_idx from original data
    - translated_phrase --> translated phrase from translation output

    """

In [0]:
def format_for_gpt(df):
    
    game = df.loc[0]['game']
    language = df.loc[0]['language']
    
    prompt = build_translation_prompt(game,language)
    prompt += "--- Phrases to Translate ---\n"
    for _,row in df.iterrows():
        row_prompt = f"English Phrase: {row['en_US']}, Character Limit: {row['target_char_limit']}, Row Idx: {row['row_idx']}\n"
        prompt += row_prompt
        
    return [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]

In [0]:
#

client = get_model_client()

def process_gpt_response(response):
    response_id = response.id
    try:
        parsed_results = json.loads(response.choices[0].message.content)
    except json.JSONDecodeError as e:
        print("⚠️ JSON parsing failed:", e)
        return pd.DataFrame([{'row_idx':None,'translated_phrase':None,'response_id':response_id}])

    parsed_response_df = pd.DataFrame(parsed_results)
    parsed_response_df['response_id'] = response_id

    #TODO: Probably good to use some data validation/sanitization here
    
    return parsed_response_df


def submit_and_process_api_call(df):
    prompt = format_for_gpt(df)
    response = client.chat.completions.create(
        model="gpt-4.1",  # or "gpt-4o" if you're using the newest multimodal model
        messages=prompt,
        temperature=0.5  # adjust for creativity vs. stability
    )
    parsed_response_df = process_gpt_response(response)
    # join with inputs?
    joined = df.merge(parsed_response_df, on='row_idx', how='left')
    return joined
    



In [0]:
parsed_output_dfs = [submit_and_process_api_call(i) for i in formatted_inputs]

In [0]:
# So each response_id corresponds to a unique game, language, platform submission

parsed_output_dfs[-1]

In [0]:
# TODO: Add logging...

# Pass this back to the 

In [0]:
# Now start modifying the 